# all-reduce-compose — worked example 1: Compose a SUM all_reduce from reduce + broadcast (injected dist)

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `all-reduce-compose`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

`all_reduce` is the composition `reduce(dst=0)` then `broadcast(src=0)`. The reduce collapses every rank's tensor onto rank 0 under some op (here SUM); the broadcast copies rank 0's now-aggregated tensor back to every rank. The result is that *all* ranks end holding the same reduced value. Using the real (or injected mock) `dist.reduce` / `dist.broadcast` keeps the drill focused on the composition shape, not on reimplementing send/recv.

## Worked solution

**Step 1 — wrap the local value in a 1-D float tensor.** Collectives operate in place on tensors, so each rank builds `tensor = t.tensor([local_value], dtype=t.float32)`. The dtype matters: `gloo` SUM on float32 is well-defined and matches what the test compares against.

**Step 2 — reduce onto rank 0 with SUM.** `dist_module.reduce(tensor, dst=0, op=dist_module.ReduceOp.SUM)` sums every rank's tensor and writes the total **only** into rank 0's tensor. Non-zero ranks are left with an unspecified (backend-dependent) value — that is why we cannot return here.

**Step 3 — broadcast from rank 0.** `dist_module.broadcast(tensor, src=0)` overwrites every rank's tensor with rank 0's copy, so the global sum now lives on all ranks. This second leg is what turns a *reduce* into an *all_reduce*.

**Step 4 — return the scalar.** `tensor.item()` is now identical on every rank. The mock `dist` in the harness implements `reduce`/`broadcast` with a thread barrier so this runs correctly on CPU.

**Why it works:** reduce gives one rank the answer; broadcast spreads that one answer everywhere. Two one-directional collectives compose into the symmetric all_reduce.

In [ ]:
import threading

class MockDist:
    class ReduceOp:
        SUM = 'sum'
        MAX = 'max'
        MIN = 'min'
        PRODUCT = 'prod'
    def __init__(self, world_size):
        self.world_size = world_size
        self._barrier = threading.Barrier(world_size)
        self._slots = [None] * world_size
        self._result = [None]
    def reduce(self, tensor, dst, op):
        rank = threading.current_thread().rank
        self._slots[rank] = tensor.clone()
        self._barrier.wait()
        if rank == dst:
            vals = t.stack(self._slots)
            if op == self.ReduceOp.SUM:
                tensor.copy_(vals.sum(dim=0))
            elif op == self.ReduceOp.MAX:
                tensor.copy_(vals.amax(dim=0))
            elif op == self.ReduceOp.MIN:
                tensor.copy_(vals.amin(dim=0))
            elif op == self.ReduceOp.PRODUCT:
                tensor.copy_(vals.prod(dim=0))
            self._result[0] = tensor.clone()
        self._barrier.wait()
    def broadcast(self, tensor, src):
        self._barrier.wait()
        tensor.copy_(self._result[0])
        self._barrier.wait()

def ex_all_reduce_sum(rank: int, world_size: int, dist_module, local_value: float) -> float:
    tensor = t.tensor([local_value], dtype=t.float32)
    dist_module.reduce(tensor, dst=0, op=dist_module.ReduceOp.SUM)
    dist_module.broadcast(tensor, src=0)
    return tensor.item()

world_size = 4
locals_ = [2.0, 5.0, 1.0, 7.0]
mock = MockDist(world_size)
results = [None] * world_size

def _run(rank):
    threading.current_thread().rank = rank
    results[rank] = ex_all_reduce_sum(rank, world_size, mock, locals_[rank])

threads = []
for r in range(world_size):
    th = threading.Thread(target=_run, args=(r,))
    th.rank = r
    threads.append(th)
for th in threads: th.start()
for th in threads: th.join()
print('per-rank results:', results)
print('all equal:', len(set(results)) == 1, '-> value', results[0], '(expected', sum(locals_), ')')